# ConvTran Reproduction: UEA Data Preparation

We prepare the 30 UEA datasets before running the baseline experiments.
This notebook downloads the archive, checks the split files and records dataset dimensions.

The BasicMotions pilot is documented in `01_reproduction.ipynb`.
Training and evaluation will be covered in `03_baseline_training.ipynb`.

Data source: https://www.timeseriesclassification.com/


In [22]:
from pathlib import Path

import pandas as pd

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)

assert ROOT is not None, "Open this notebook from the project repository."

DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
ARCHIVE_DIR = ROOT / "Dataset" / "archives"
RESULTS_DIR = ROOT / "results"

print("Repository:", ROOT)
print("Data directory:", DATA_ROOT)


Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Data directory: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/Dataset/UEA/Multivariate_ts


## Dataset list

We use the 30 datasets from the UEA multivariate archive.


In [23]:
UEA_DATASETS = [
    "ArticularyWordRecognition",
    "AtrialFibrillation",
    "BasicMotions",
    "CharacterTrajectories",
    "Cricket",
    "DuckDuckGeese",
    "EigenWorms",
    "Epilepsy",
    "ERing",
    "EthanolConcentration",
    "FaceDetection",
    "FingerMovements",
    "HandMovementDirection",
    "Handwriting",
    "Heartbeat",
    "InsectWingbeat",
    "JapaneseVowels",
    "Libras",
    "LSST",
    "MotorImagery",
    "NATOPS",
    "PenDigits",
    "PEMS-SF",
    "PhonemeSpectra",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "SpokenArabicDigits",
    "StandWalkJump",
    "UWaveGestureLibrary",
]

assert len(UEA_DATASETS) == len(set(UEA_DATASETS)) == 30
print("Expected datasets:", len(UEA_DATASETS))


Expected datasets: 30


## Download and extraction

We download the archive from the Time Series Classification website.
The ZIP is kept locally so it can be reused.

Existing split files are preserved, including those used in the
BasicMotions pilot. New files are extracted to the layout expected
by our baseline runner.


In [24]:
import shutil
import zipfile

import requests
from tqdm.auto import tqdm

ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
archive_path = ARCHIVE_DIR / "Multivariate2018_ts.zip"
archive_url = (
    "https://www.timeseriesclassification.com/"
    "aeon-toolkit/Archives/Multivariate2018_ts.zip"
)

if not archive_path.exists():
    partial = archive_path.with_suffix(".zip.part")

    with requests.get(
        archive_url, stream=True, timeout=(30, 120)
    ) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with (
            partial.open("wb") as target,
            tqdm(
                total=total or None,
                unit="B",
                unit_scale=True,
                desc="Downloading UEA",
            ) as progress,
        ):
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    target.write(chunk)
                    progress.update(len(chunk))

    if not zipfile.is_zipfile(partial):
        raise ValueError("The downloaded file is not a valid ZIP archive.")

    partial.replace(archive_path)
else:
    print("Using existing archive:", archive_path)

with zipfile.ZipFile(archive_path) as archive:
    members = {}

    for name in UEA_DATASETS:
        for split in ("TRAIN", "TEST"):
            filename = f"{name}_{split}.ts"
            matches = [
                member for member in archive.namelist()
                if Path(member).name == filename
            ]

            if len(matches) != 1:
                raise ValueError(
                    f"Expected one archive member for {filename}, "
                    f"found {len(matches)}."
                )

            members[(name, filename)] = matches[0]

    extracted = 0
    preserved = 0

    for (name, filename), member in tqdm(
        members.items(), desc="Extracting split files"
    ):
        destination = DATA_ROOT / name / filename

        if destination.exists():
            preserved += 1
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_suffix(".ts.part")

        with archive.open(member) as source, partial.open("wb") as target:
            shutil.copyfileobj(source, target)

        partial.replace(destination)
        extracted += 1

print(f"Extracted files: {extracted}")
print(f"Preserved files: {preserved}")


Using existing archive: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/Dataset/archives/Multivariate2018_ts.zip


Extracting split files: 100%|██████████| 60/60 [00:00<00:00, 111699.17it/s]

Extracted files: 0
Preserved files: 60


## Split-file availability

Each dataset must have a non-empty TRAIN file and TEST file.
This check confirms file availability; it does not establish that training will fit in GPU memory.


In [25]:
inventory_rows = []

for name in UEA_DATASETS:
    train = DATA_ROOT / name / f"{name}_TRAIN.ts"
    test = DATA_ROOT / name / f"{name}_TEST.ts"

    inventory_rows.append({
        "dataset": name,
        "train_file": train.is_file() and train.stat().st_size > 0,
        "test_file": test.is_file() and test.stat().st_size > 0,
    })

inventory = pd.DataFrame(inventory_rows)
inventory["files_present"] = (
    inventory["train_file"] & inventory["test_file"]
)

print(
    f"Datasets with both files: "
    f"{inventory['files_present'].sum()}/{len(UEA_DATASETS)}"
)

assert inventory["files_present"].all(), "Some split files are missing or empty."
inventory


Datasets with both files: 30/30


,dataset,train_file,test_file,files_present
0,ArticularyWordRecognition,True,True,True
1,AtrialFibrillation,True,True,True
2,BasicMotions,True,True,True
3,CharacterTrajectories,True,True,True
4,Cricket,True,True,True
5,DuckDuckGeese,True,True,True
6,EigenWorms,True,True,True
7,Epilepsy,True,True,True
8,ERing,True,True,True
9,EthanolConcentration,True,True,True


## Dataset dimensions

We read each file sequentially to count samples, channels and sequence
lengths without loading the full dataset into memory.

Lengths are measured across all channels in both splits. The parser
expects classification files without explicit timestamps.

This step does not create or modify the upstream preprocessing caches.


In [26]:
def inspect_ts_file(path):
    samples = 0
    channel_counts = set()
    classes = set()
    min_length = None
    max_length = 0
    in_data = False

    with path.open(encoding="utf-8-sig") as stream:
        for line in stream:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if not in_data:
                lower = line.lower()

                if lower.startswith("@timestamps"):
                    if lower.split()[1] != "false":
                        raise ValueError(
                            f"Explicit timestamps are not supported: {path}"
                        )

                if lower.startswith("@classlabel"):
                    if lower.split()[1] != "true":
                        raise ValueError(f"Expected classification labels: {path}")

                if lower == "@data":
                    in_data = True

                continue

            parts = line.split(":")
            if len(parts) < 2:
                raise ValueError(f"Invalid sample in {path}")

            channels = parts[:-1]
            label = parts[-1].strip()
            channel_counts.add(len(channels))
            classes.add(label)

            for channel in channels:
                length = channel.count(",") + 1 if channel.strip() else 0
                min_length = (
                    length if min_length is None
                    else min(min_length, length)
                )
                max_length = max(max_length, length)

            samples += 1

    if not in_data or samples == 0:
        raise ValueError(f"No samples found: {path}")

    if len(channel_counts) != 1:
        raise ValueError(f"Inconsistent channel counts: {path}")

    return {
        "samples": samples,
        "channels": channel_counts.pop(),
        "min_length": min_length,
        "max_length": max_length,
        "classes": classes,
    }


dimension_rows = []

for name in tqdm(UEA_DATASETS, desc="Inspecting datasets"):
    folder = DATA_ROOT / name
    train = inspect_ts_file(folder / f"{name}_TRAIN.ts")
    test = inspect_ts_file(folder / f"{name}_TEST.ts")

    assert train["channels"] == test["channels"], (
        f"Train/test channel mismatch: {name}"
    )

    dimension_rows.append({
        "dataset": name,
        "train_samples": train["samples"],
        "test_samples": test["samples"],
        "channels": train["channels"],
        "classes": len(train["classes"]),
        "min_length": min(train["min_length"], test["min_length"]),
        "max_length": max(train["max_length"], test["max_length"]),
        "same_class_set": train["classes"] == test["classes"],
    })

dimensions = pd.DataFrame(dimension_rows)
dimensions["variable_length"] = (
    dimensions["min_length"] != dimensions["max_length"]
)

dimensions.sort_values("max_length", ascending=False).reset_index(drop=True)


Inspecting datasets: 100%|██████████| 30/30 [00:05<00:00,  5.18it/s]


,dataset,train_samples,test_samples,channels,classes,min_length,max_length,same_class_set,variable_length
0,EigenWorms,128,131,6,5,17984,17984,True,False
1,MotorImagery,278,100,64,2,3000,3000,True,False
2,StandWalkJump,12,15,4,3,2500,2500,True,False
3,EthanolConcentration,261,263,3,4,1751,1751,True,False
4,Cricket,108,72,6,12,1197,1197,True,False
5,SelfRegulationSCP2,200,180,7,2,1152,1152,True,False
6,SelfRegulationSCP1,268,293,6,2,896,896,True,False
7,AtrialFibrillation,15,15,2,3,640,640,True,False
8,Heartbeat,204,205,61,2,405,405,True,False
9,HandMovementDirection,160,74,10,4,400,400,True,False


## Saved metadata

We save the measured dimensions for use in the training notebook.
This table contains dataset metadata, not model results.


In [27]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
destination = RESULTS_DIR / "dataset_dimensions.csv"

dimensions.to_csv(destination, index=False)
print("Saved:", destination)

dimensions.sort_values("max_length", ascending=False).head(10)


Saved: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/dataset_dimensions.csv


,dataset,train_samples,test_samples,channels,classes,min_length,max_length,same_class_set,variable_length
6,EigenWorms,128,131,6,5,17984,17984,True,False
19,MotorImagery,278,100,64,2,3000,3000,True,False
28,StandWalkJump,12,15,4,3,2500,2500,True,False
9,EthanolConcentration,261,263,3,4,1751,1751,True,False
4,Cricket,108,72,6,12,1197,1197,True,False
26,SelfRegulationSCP2,200,180,7,2,1152,1152,True,False
25,SelfRegulationSCP1,268,293,6,2,896,896,True,False
1,AtrialFibrillation,15,15,2,3,640,640,True,False
14,Heartbeat,204,205,61,2,405,405,True,False
12,HandMovementDirection,160,74,10,4,400,400,True,False


## Next step

We will use these measurements to plan the baseline runs in
`03_baseline_training.ipynb`.

Sequence length is relevant because the released attention implementation
creates matrices whose size grows quadratically with length. Dimensions
alone do not provide a complete GPU memory estimate.

Long sequences, missing values and variable lengths may require additional
checks. Any changes to the baseline will be recorded with the corresponding
experiments.
